# 섹션3-5. Python과 ChatGPT를 활용한 박스플롯, 히스토그램, ECDF

> 강의: [32가지 데이터 시각화 전략 - 비전공자를 위한 기초이론 & 실습](https://www.inflearn.com/course/32-data-visualizatio/dashboard?cid=343563) (반병현) — 전체 16강

- [x] 강의 시청 완료
- [x] 실습/정리 완료

## 배운 내용

<!-- 강의를 보면서 핵심을 적는다 -->

-

## 목표 / 재현할 것

<!-- 이 강의에서 만든 차트를 내 방식대로 다시 만들어본다 -->

-


## 실습

16강은 VS Code에서 파이썬 스크립트(`main.py`)를 AI와 함께 완성해가는 과정이었다.
포트폴리오에는 그 스크립트를 실제로 실행한 결과를 [Python을 사용한 시각화](../docs/cases/python-viz/)로
정리해뒀고, 여기서는 **그 결과가 왜 페이지의 핵심 논지가 됐는지**를 다시 재현한다.

데이터는 field2scene(개인 프로젝트)의 24쌍 실촬영/렌더 검출 수 — 실제 CSV는
`github.com/leeyunhome/field2scene/report/gap_tomato/per_frame.csv`에 공개되어 있고,
분량이 작아 여기 직접 옮겨 적었다.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_utils import setup, load_sample

setup()
rng = np.random.default_rng(0)

# field2scene report/gap_tomato/per_frame.csv — 24개 view쌍의 검출 수
REAL = [12, 11, 12, 14, 15, 11, 12, 10, 9, 11, 8, 7, 9, 7, 6, 5, 6, 5, 4, 3, 3, 2, 2, 3]
REND = [12, 11, 11, 13, 14, 11, 12, 9, 9, 11, 8, 7, 8, 8, 6, 5, 4, 3, 3, 3, 2, 1, 1, 3]
assert sum(REAL) == 187 and sum(REND) == 175  # 리포트 총계와 대조

df = pd.DataFrame({"검출수": REAL + REND, "촬영방식": ["실촬영"] * 24 + ["3DGS 렌더"] * 24})
df.head()

### `main.py`가 만드는 세 가지 — 박스플롯 · 히스토그램 · ECDF

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.3))

groups = [df.loc[df["촬영방식"] == g, "검출수"].values for g in ["실촬영", "3DGS 렌더"]]

axes[0].boxplot(groups, tick_labels=["실촬영", "3DGS 렌더"])
axes[0].set_title("박스플롯")

for g, vals in zip(["실촬영", "3DGS 렌더"], groups):
    axes[1].hist(vals, bins=8, alpha=0.5, label=g)
axes[1].legend(fontsize=8)
axes[1].set_title("히스토그램")

for g, vals in zip(["실촬영", "3DGS 렌더"], groups):
    xs = np.sort(vals)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    axes[2].step(xs, ys, where="post", label=g)
axes[2].legend(fontsize=8)
axes[2].set_title("ECDF")

fig.suptitle("세 그래프 모두 — 두 그룹이 비슷해 보인다")
plt.tight_layout()
plt.show()

print(f"중앙값  실촬영 {np.median(groups[0])}  렌더 {np.median(groups[1])}")
print(f"평균    실촬영 {groups[0].mean():.2f}  렌더 {groups[1].mean():.2f}")

> 세 그래프 모두 두 그룹을 **독립 표본**으로 취급한다. 그런데 이 24개는 독립이 아니라
> **같은 view를 두 방식으로 찍은 쌍**이다. 짝 정보를 버리면 무엇을 놓치는지 아래서 확인한다.

### 짝을 살리면 — 같은 데이터, 반대 결론

In [ ]:
d = np.array(REND) - np.array(REAL)
n_down, n_same, n_up = (d < 0).sum(), (d == 0).sum(), (d > 0).sum()
print(f"렌더가 적은 쌍 {n_down} / 같은 쌍 {n_same} / 많은 쌍 {n_up}  (총 24쌍)")

from scipy.stats import wilcoxon
stat, p = wilcoxon(REAL, REND)
print(f"Wilcoxon 부호순위 검정 p = {p:.4f}")

fig, ax = plt.subplots(figsize=(5, 4.3))
for a, r in zip(REAL, REND):
    color = "#E45756" if r < a else ("#4C78A8" if r > a else "gray")
    ax.plot([0, 1], [a, r], color=color, alpha=0.7, lw=1.3)
ax.set_xticks([0, 1], ["실촬영", "3DGS 렌더"])
ax.set_ylabel("검출 수")
ax.set_title(f"쌍별 연결선 — {n_down}쌍 감소 / {n_up}쌍 증가")
plt.show()

**결론.** 박스플롯·히스토그램·ECDF 셋 다 "비슷하다"로 읽히지만, 쌍으로 세면
24쌍 중 23쌍이 같거나 줄었고 늘어난 건 1쌍뿐이다(p=0.0049). **평균이 비슷한 것과
차이가 없는 것은 다르다** — 이 구분이 16강 실습에서 얻은 가장 중요한 교훈이다.

### 코드에서 신경 쓴 것 (`main.py`)

- 한글 CSV는 UTF-8이 아니라 CP949인 경우가 많아 인코딩을 순서대로 시도
- seaborn이 없어도 matplotlib으로 같은 그림을 그리도록 폴백 처리
- `plt.rcParams['axes.unicode_minus'] = False` — 한글 폰트에서 음수 부호가 깨지는 문제


---

## 메모

-
